# Uber Pickups in NYC - Machine Learning Pipeline

## Competencies evaluated:
1. Process structured data using Scikit-Learn (encoding, scaling, splitting)
2. Predictive analysis (Supervised Learning)
3. Clustering / Dimensionality reduction (Unsupervised Learning)
4. Evaluate predictive performance and feature importance

## Overview
This notebook demonstrates a complete ML workflow to recommend hot-zones for Uber drivers in New York City based on past pickup data.


In [ ]:
%pip install pandas numpy plotly scikit-learn nbformat


In [ ]:
pip install tensorflow tensorflow_hub sentence-transformers scikit-learn seaborn --break-system-packages 2>&1 | tail -5 install pandas numpy plotly scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import urllib.request
import zipfile
import io
import plotly.express as px
import plotly.graph_objects as go
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans, DBSCAN
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')


## 1. Data Integration
We will download the ZIP dataset and extract the raw data for April 2014 directly into a Pandas DataFrame. We sample down to 100,000 rows to speed up Exploratory Data Analysis and Model Training.


In [ ]:
url = 'https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Machine+Learning+non+Supervis%C3%A9/Projects/uber-trip-data.zip'
print(f'Downloading dataset from {url}...')
req = urllib.request.urlopen(url)
zip_file = zipfile.ZipFile(io.BytesIO(req.read()))

filename = 'uber-trip-data/uber-raw-data-apr14.csv'
with zip_file.open(filename) as f:
    df = pd.read_csv(f)

df = df.sample(n=100000, random_state=42).reset_index(drop=True)
display(df.head())


## 2. Exploratory Data Analysis (EDA)
Before building machine learning models, it is essential to understand the shape of our data. We will check for missing values, analyze the distribution of pickups over time, and visualize the spatial distribution of rides.


In [ ]:
# Check dataset information and missing values
df.info()
display(df.describe())


### Feature Engineering for EDA
We extract temporal features from the 'Date/Time' column to see exactly when people use Uber the most.


In [ ]:
df['Date/Time'] = pd.to_datetime(df['Date/Time'])
df['Hour'] = df['Date/Time'].dt.hour
df['DayOfWeek'] = df['Date/Time'].dt.dayofweek
df['DayName'] = df['Date/Time'].dt.day_name()
display(df.head())


### Temporal Distribution
When are the pickups happening? Let's plot the volume of rides per Hour and per Day of the Week.


In [ ]:
fig1 = px.histogram(df, x='Hour', marginal='box', title='Distribution of Pickups per Hour')
fig1.show()

fig2 = px.histogram(df, x='DayName', category_orders={'DayName': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']}, title='Distribution of Pickups per Day of the Week')
fig2.show()


### Spatial Distribution
Let's plot the raw coordinates to see the geographical density of Manhattan and surrounding boroughs.


In [ ]:
fig3 = px.scatter(df, x='Lon', y='Lat', opacity=0.1, title='Raw Scatter Plot of Pickups (Lat vs Lon)', width=800, height=800)
fig3.show()


## 3. Unsupervised Learning (Clustering - Hot Zones)
Based on our EDA, rides are highly localized geographically. We can use unsupervised machine learning (KMeans and DBSCAN) to find distinct "Hot Zones" (clusters) where pickups are densely packed.


In [ ]:
# Using a smaller sample to speed up DBSCAN which is computationally expensive
sample_df = df.sample(n=10000, random_state=42).copy()
X_geo = sample_df[['Lat', 'Lon']]

# KMeans
kmeans = KMeans(n_clusters=10, random_state=42)
sample_df['Cluster_KMeans'] = kmeans.fit_predict(X_geo)

# DBSCAN
dbscan = DBSCAN(eps=0.01, min_samples=50)
sample_df['Cluster_DBSCAN'] = dbscan.fit_predict(X_geo)

df['HotZone'] = kmeans.predict(df[['Lat', 'Lon']])
print('Clustering computed.')


Let's visualize the resulting 10 Hot Zones using a Mapbox projection overlay.


In [ ]:
fig = px.scatter_mapbox(
    sample_df, lat='Lat', lon='Lon',
    color='Cluster_KMeans',
    color_continuous_scale=px.colors.cyclical.IceFire,
    zoom=10, mapbox_style='carto-positron',
    title='Uber Pickups - 10 KMeans Hot Zones in NYC'
)
fig.update_layout(coloraxis_showscale=False)
fig.show()


## 4. Supervised Learning (Predictive Analysis)
Now that we know *where* the Hot Zones are, we need to advise drivers on where to go based on the current *time*. We will predict the `HotZone` (target variable) using `Hour`, `DayOfWeek`, and the Uber `Base` categorical feature.


In [ ]:
X = df[['Hour', 'DayOfWeek', 'Base']]
y = df['HotZone']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

numeric_features = ['Hour', 'DayOfWeek']
categorical_features = ['Base']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('scaler', StandardScaler())]), numeric_features),
        ('cat', Pipeline(steps=[('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))]), categorical_features)
    ])

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1))
])

print('Training the Random Forest model...')
pipeline.fit(X_train, y_train)
print('Model Trained.')


## 5. Model Evaluation and Feature Importance
Let's evaluate how well our model predicts the correct Hot Zone and which features are driving these predictions.


In [ ]:
y_pred = pipeline.predict(X_test)
print(f'Model Accuracy: {accuracy_score(y_test, y_pred):.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred))


In [ ]:
# Feature Importance
cat_encoder = pipeline.named_steps['preprocessor'].named_transformers_['cat'].named_steps['encoder']
cat_names = cat_encoder.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + cat_names

importances = pipeline.named_steps['classifier'].feature_importances_
feature_importance_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances}).sort_values('Importance', ascending=False)

fig_imp = px.bar(feature_importance_df, x='Feature', y='Importance', title='Feature Importances driving Hot Zone Prediction')
fig_imp.show()
display(feature_importance_df)
